# Thermal limits of earthing conductors (IEC 60949 / IEC 60909-0)

Does an earthing conductor survive the fault current *thermally*? This is the
equipment-integrity counterpart to the touch-voltage (person-safety)
assessment.

- **IEC 60909-0**: thermally equivalent short-time current
  $I_{th} = I_{s,\mathrm{RMS}} \cdot \sqrt{m + n}$, with the heat factor $m$
  of the decaying DC component and $n$ of the AC component.
- **IEC 60949**: admissible adiabatic current $I_{adm} = k \cdot S / \sqrt{t_k}$,
  with the material constant $k$ from the initial and final temperature.

The linear AC RMS shield currents are superposed by the solve as usual; the
non-linear IEC 60909 factor is applied to that aggregate ($I_{th}$ is never
superposed directly).

In [ ]:
import groundinsight as gi

gi.IEC60949_MATERIALS

The material constant $k = K\sqrt{\ln\big((\theta_f+\beta)/(\theta_i+\beta)\big)}$
reproduces the tables of the standard:

In [ ]:
print("Cu XLPE (90->250 C):", round(gi.iec60949_k("Cu", 90, 250), 1), "(table: 143)")
print("Cu PVC  (70->160 C):", round(gi.iec60949_k("Cu", 70, 160), 1), "(table: 115)")
print("Al XLPE (90->250 C):", round(gi.iec60949_k("Al", 90, 250), 1), "(table: 94)")

## A small network with a copper earthing conductor

Two 20 kV buses, a 5000 A earth fault, and a `BranchType` that carries the
conductor material, its cross-section and the initial and final temperatures
of the thermal check. The impedances are illustrative.

In [ ]:
def build(cross_section_mm2):
    net = gi.create_network(name="thermal", frequencies=[50.0])
    bt = gi.BusType(name="b", system_type="s", voltage_level=20.0, impedance_formula="rho*0 + 0.5")
    ct = gi.BranchType(name="c", grounding_conductor=True,
                       self_impedance_formula="(rho*0 + 0.25 + j*f*0.012)*l",
                       mutual_impedance_formula="(rho*0 + 0.05 + j*f*0.012)*l",
                       conductor_material="Cu", cross_section_mm2=cross_section_mm2,
                       theta_initial_C=20.0, theta_final_C=300.0)
    for b in ["b1", "b2"]:
        gi.create_bus(name=b, type=bt, network=net)
    gi.create_branch(name="c1", type=ct, from_bus="b1", to_bus="b2", length=1.0, network=net)
    gi.create_source(name="src", bus="b1", values={50.0: 5000.0}, network=net)
    gi.create_fault(name="F", bus="b2", network=net)
    gi.run_fault(net, fault="F")
    return net

## Check a 16 mm² conductor, fault duration 1 s, R/X = 0.1

`t_k_s` is the short-circuit duration in seconds; `r_to_x` the R/X ratio of
the source, from which the peak factor $\kappa$ and the DC heat factor $m$
follow.

In [ ]:
net = build(cross_section_mm2=16.0)
df = gi.check_conductor_limits(net, "F", t_k_s=1.0, r_to_x=0.1)
df.select("branch_name", "I_s_rms_A", "kappa", "m", "n", "I_th_A", "k",
          "I_admissible_A", "utilization", "within_limit")

`within_limit=False` and `utilization > 1` mean the conductor is thermally
undersized.

**The same two numbers by hand**, from the shield current of the solve and the
formulas of the two standards:

In [ ]:
import math

row = df.to_dicts()[0]
kappa = gi.kappa_from_r_to_x(0.1)                     # 1.02 + 0.98 exp(-3 R/X)
m = gi.iec60909_m(kappa, 50.0, 1.0)                   # DC heat factor
i_th = row["I_s_rms_A"] * math.sqrt(m + 1.0)          # n = 1, far from generators
k = gi.iec60949_k("Cu", 20.0, 300.0)
i_adm = k * 16.0 / math.sqrt(1.0)
print(f"kappa = {kappa:.6f}  (1.02 + 0.98 e^-0.3 = {1.02 + 0.98 * math.exp(-0.3):.6f})")
print(f"I_th  = {i_th:.1f} A   check_conductor_limits: {row['I_th_A']:.1f} A")
print(f"I_adm = {i_adm:.1f} A   check_conductor_limits: {row['I_admissible_A']:.1f} A")

Increase the cross-section and the utilization drops below 1:

In [ ]:
for S in [16.0, 50.0, 95.0, 240.0]:
    r = gi.check_conductor_limits(build(S), "F", t_k_s=1.0, r_to_x=0.1).to_dicts()[0]
    print(f"S={S:6.1f} mm^2  I_th={r['I_th_A']:7.0f} A  I_adm={r['I_admissible_A']:8.0f} A  "
          f"util={r['utilization']:.2f}  ok={r['within_limit']}")

## Fault duration matters (adiabatic ∝ 1/√t_k)

A longer clearing time lowers the admissible current, so the same conductor
may fail; the DC heat factor $m$ falls at the same time, because the DC offset
has decayed for a larger part of the fault:

In [ ]:
for t_k in [0.1, 0.5, 1.0, 3.0]:
    r = gi.check_conductor_limits(build(50.0), "F", t_k_s=t_k, r_to_x=0.1).to_dicts()[0]
    print(f"t_k={t_k:4.1f} s  m={r['m']:.3f}  I_adm={r['I_admissible_A']:8.0f} A  "
          f"util={r['utilization']:.2f}  ok={r['within_limit']}")

## Notes

- `kappa` (peak factor) comes from `r_to_x` via $1.02 + 0.98\,e^{-3R/X}$, or is
  passed directly.
- `n = 1` is the far-from-generator default (distribution earthing); pass
  `n_factor` otherwise.
- `gi.check_node_limits` checks the earthing conductor and the electrode of
  every bus in the same way, and `gi.read_shortcircuit_results` takes
  $\kappa$ and the short-circuit current from a pandapower `calc_sc` run
  ([Analysis](../../api/analysis/#node-thermal-limits-earthing-conductor-vs-earth-electrode),
  [I/O](../../api/io/#short-circuit-characteristics-iec-60909-0)).
- The check is adiabatic: the heat given off during the fault is neglected,
  which underestimates the admissible current and is on the safe side
  ([Standards and scope](../../explanation/standards/)).

References: IEC 60909-0:2016, IEC 60949:1988, IEC 60364-5-54, EN 50522.